# 12 · Modelos clásicos

**Proyecto:** EmoGol — Trabajo de grado, Universidad del Cauca  
**Autor:** Santiago Fernando Ortega Segura · **Director:** Ph.D. Oscar Mauricio Caicedo Rendón

Entrena y evalúa 6 modelos clásicos con los mismos datos: SVM lineal (TF-IDF de 1 y 2 palabras), árbol de decisión y Random Forest (TF-IDF + SVD de 300 componentes), CNN, BiLSTM y GNN (TextGCN).

*Nombre durante el desarrollo: `08_Clasicos_Split_Oficial` (algunas salidas conservan esa numeración).*

## 1. Instalar dependencias y descargar el split oficial

In [1]:
!pip install -q -U "pandas<3" scikit-learn scipy nltk "huggingface_hub<2.0" torch tqdm

from google.colab import userdata
from huggingface_hub import login, hf_hub_download

login(token=userdata.get('HF_TOKEN'))

TU_USUARIO_HF = "santiagortegasegura"
REPO_DATOS = f"{TU_USUARIO_HF}/emotion-futbol-data"

import pandas as pd
import numpy as np

ruta_train = hf_hub_download(repo_id=REPO_DATOS, repo_type="dataset", filename="04b_entrenamiento_72k.csv")
ruta_val_interna = hf_hub_download(repo_id=REPO_DATOS, repo_type="dataset", filename="04b_validacion_interna_18k.csv")
ruta_test = hf_hub_download(repo_id=REPO_DATOS, repo_type="dataset", filename="final/prueba_final_10k.csv")

df_train = pd.read_csv(ruta_train)
df_val_interna = pd.read_csv(ruta_val_interna)
df_test = pd.read_csv(ruta_test)

print(f"Entrenamiento: {len(df_train)} | Validación interna: {len(df_val_interna)} | Test final: {len(df_test)}")

SEED = 42
np.random.seed(SEED)
EMOCIONES = ['alegria', 'tristeza', 'enojo', 'miedo', 'sorpresa', 'burla', 'neutral']
label2id = {e: i for i, e in enumerate(EMOCIONES)}
id2label = {i: e for i, e in enumerate(EMOCIONES)}

COLUMNA_TEXTO = "text"
y_train = df_train['label'].values
y_test = df_test['label'].values

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 91.2/91.2 kB 5.1 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 62.4/62.4 kB 4.5 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 57.4/57.4 kB 5.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.3/12.3 MB 104.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.1/9.1 MB 127.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 35.3/35.3 MB 20.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.8/1.8 MB 75.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 846.4/846.4 kB 59.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 554.6/554.6 MB 859.5 kB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 553.1/553.1 MB 2.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 170.1/170.1 MB 6.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 216.0/216.0 MB 5.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 

04b_entrenamiento_72k.csv:   0%|          | 0.00/8.65M [00:00<?, ?B/s]

04b_validacion_interna_18k.csv:   0%|          | 0.00/2.16M [00:00<?, ?B/s]

prueba_final_10k.csv:   0%|          | 0.00/1.19M [00:00<?, ?B/s]

Entrenamiento: 72000 | Validación interna: 18000 | Test final: 10000


## 2. Función de registro de resultados

In [2]:
import os, time, pickle
from sklearn.metrics import classification_report

RUTA_RESULTADOS = "08_resultados_clasicos.csv"

def registrar_resultado(nombre, y_true, y_pred, latencia_prom_seg=None, memoria_pico_mb=None,
                         almacenamiento_modelo_mb=None, n_test=None, nota=""):
    reporte = classification_report(y_true, y_pred, labels=EMOCIONES, output_dict=True, zero_division=0)
    fila = {
        "modelo": nombre,
        "n_test": n_test if n_test is not None else len(y_true),
        "accuracy": reporte["accuracy"],
        "precision_macro": reporte["macro avg"]["precision"],
        "recall_macro": reporte["macro avg"]["recall"],
        "f1_macro": reporte["macro avg"]["f1-score"],
        "precision_weighted": reporte["weighted avg"]["precision"],
        "recall_weighted": reporte["weighted avg"]["recall"],
        "f1_weighted": reporte["weighted avg"]["f1-score"],
        "latencia_prom_seg": latencia_prom_seg,
        "memoria_pico_mb": memoria_pico_mb,
        "almacenamiento_modelo_mb": almacenamiento_modelo_mb,
        "nota": nota,
    }
    for e in EMOCIONES:
        fila[f"f1_{e}"] = reporte.get(e, {}).get("f1-score", None)

    df_fila = pd.DataFrame([fila])
    if os.path.exists(RUTA_RESULTADOS):
        df_res = pd.read_csv(RUTA_RESULTADOS)
        df_res = df_res[df_res['modelo'] != nombre]
        df_res = pd.concat([df_res, df_fila], ignore_index=True)
    else:
        df_res = df_fila
    df_res.to_csv(RUTA_RESULTADOS, index=False)

    print(classification_report(y_true, y_pred, labels=EMOCIONES, zero_division=0))
    print(f"Guardado en {RUTA_RESULTADOS}. Modelos registrados: {sorted(df_res['modelo'].tolist())}")

def tamano_mb(obj):
    return len(pickle.dumps(obj)) / (1024 ** 2)

## 3. TF-IDF + SVD para árbol/Random Forest

In [3]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.decomposition import TruncatedSVD

STOPWORDS_ES = None
try:
    import nltk
    nltk.download('stopwords', quiet=True)
    from nltk.corpus import stopwords
    STOPWORDS_ES = stopwords.words('spanish')
except Exception:
    print("No se pudieron cargar stopwords de nltk -- se sigue sin lista de stopwords (no es crítico).")

tfidf = TfidfVectorizer(max_features=30000, ngram_range=(1, 2), stop_words=STOPWORDS_ES, min_df=2)
X_train_tfidf = tfidf.fit_transform(df_train[COLUMNA_TEXTO])
X_test_tfidf = tfidf.transform(df_test[COLUMNA_TEXTO])
print(f"TF-IDF: {X_train_tfidf.shape[1]} features")

svd = TruncatedSVD(n_components=300, random_state=SEED)
X_train_svd = svd.fit_transform(X_train_tfidf)
X_test_svd = svd.transform(X_test_tfidf)
print(f"Varianza explicada por los 300 componentes SVD: {svd.explained_variance_ratio_.sum():.1%}")

[nltk_data] Error loading stopwords: Security Violation
[nltk_data]     [pathsec.urlopen]: refusing a proxied fetch of
[nltk_data]     'https://raw.githubusercontent.com/nltk/nltk_data/gh-
[nltk_data]     pages/index.xml'. A configured proxy performs the
[nltk_data]     egress, so NLTK cannot pin the validated IP and SSRF
[nltk_data]     protection cannot be enforced (CWE-918). If and only
[nltk_data]     if the proxy is trusted to be SSRF-safe, opt in via
[nltk_data]     NLTK_ALLOW_PROXIED_URLOPEN=1 or
[nltk_data]     nltk.pathsec.ALLOW_PROXIED_FETCH=True.


No se pudieron cargar stopwords de nltk -- se sigue sin lista de stopwords (no es crítico).
TF-IDF: 30000 features
Varianza explicada por los 300 componentes SVD: 23.9%


## 4. SVM (LinearSVC)

In [4]:
from sklearn.svm import LinearSVC

svm = LinearSVC(class_weight='balanced', random_state=SEED, max_iter=5000)

inicio = time.time()
svm.fit(X_train_tfidf, y_train)
print(f"Entrenamiento SVM: {time.time() - inicio:.1f} s")

inicio_inf = time.time()
pred_svm = svm.predict(X_test_tfidf)
latencia_svm = (time.time() - inicio_inf) / len(df_test)

print("=== SVM (LinearSVC, TF-IDF, split oficial) ===")
registrar_resultado("SVM", y_test, pred_svm, latencia_prom_seg=latencia_svm,
                     almacenamiento_modelo_mb=tamano_mb(svm))

Entrenamiento SVM: 11.5 s
=== SVM (LinearSVC, TF-IDF, split oficial) ===
              precision    recall  f1-score   support

     alegria       0.74      0.71      0.72      3282
    tristeza       0.47      0.54      0.50       479
       enojo       0.68      0.67      0.67      3104
       miedo       0.27      0.31      0.29        26
    sorpresa       0.26      0.33      0.29       409
       burla       0.61      0.53      0.56      1633
     neutral       0.38      0.47      0.42      1067

    accuracy                           0.62     10000
   macro avg       0.49      0.51      0.49     10000
weighted avg       0.63      0.62      0.62     10000

Guardado en 08_resultados_clasicos.csv. Modelos registrados: ['SVM']


## 5. Árbol de decisión

In [5]:
from sklearn.tree import DecisionTreeClassifier

arbol = DecisionTreeClassifier(class_weight='balanced', random_state=SEED, max_depth=30, min_samples_leaf=5)

inicio = time.time()
arbol.fit(X_train_svd, y_train)
print(f"Entrenamiento árbol: {time.time() - inicio:.1f} s")

inicio_inf = time.time()
pred_arbol = arbol.predict(X_test_svd)
latencia_arbol = (time.time() - inicio_inf) / len(df_test)

print("=== Árbol de decisión (SVD, split oficial) ===")
registrar_resultado("Arbol de decision", y_test, pred_arbol, latencia_prom_seg=latencia_arbol,
                     almacenamiento_modelo_mb=tamano_mb(arbol))

Entrenamiento árbol: 75.0 s
=== Árbol de decisión (SVD, split oficial) ===
              precision    recall  f1-score   support

     alegria       0.52      0.32      0.40      3282
    tristeza       0.10      0.21      0.14       479
       enojo       0.42      0.32      0.37      3104
       miedo       0.05      0.12      0.07        26
    sorpresa       0.07      0.20      0.11       409
       burla       0.30      0.33      0.31      1633
     neutral       0.17      0.26      0.20      1067

    accuracy                           0.31     10000
   macro avg       0.23      0.25      0.23     10000
weighted avg       0.38      0.31      0.33     10000

Guardado en 08_resultados_clasicos.csv. Modelos registrados: ['Arbol de decision', 'SVM']


## 6. Random Forest

In [6]:
from sklearn.ensemble import RandomForestClassifier

rf = RandomForestClassifier(n_estimators=200, class_weight='balanced', random_state=SEED, n_jobs=-1, max_depth=30)

inicio = time.time()
rf.fit(X_train_svd, y_train)
print(f"Entrenamiento Random Forest: {time.time() - inicio:.1f} s")

inicio_inf = time.time()
pred_rf = rf.predict(X_test_svd)
latencia_rf = (time.time() - inicio_inf) / len(df_test)

print("=== Random Forest (SVD, split oficial) ===")
registrar_resultado("Random Forest", y_test, pred_rf, latencia_prom_seg=latencia_rf,
                     almacenamiento_modelo_mb=tamano_mb(rf))

Entrenamiento Random Forest: 268.4 s
=== Random Forest (SVD, split oficial) ===
              precision    recall  f1-score   support

     alegria       0.60      0.61      0.60      3282
    tristeza       0.51      0.17      0.26       479
       enojo       0.49      0.69      0.57      3104
       miedo       0.33      0.12      0.17        26
    sorpresa       0.29      0.08      0.13       409
       burla       0.52      0.33      0.41      1633
     neutral       0.37      0.31      0.34      1067

    accuracy                           0.52     10000
   macro avg       0.44      0.33      0.35     10000
weighted avg       0.51      0.52      0.50     10000

Guardado en 08_resultados_clasicos.csv. Modelos registrados: ['Arbol de decision', 'Random Forest', 'SVM']


## 7. Vocabulario y datasets para CNN/RNN (PyTorch)

In [7]:
import re
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from collections import Counter
from sklearn.metrics import f1_score

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Usando: {DEVICE}")

def tokenizar(texto):
    return re.findall(r"\w+|[^\w\s]", str(texto).lower(), re.UNICODE)

MAX_VOCAB = 20000
MAX_LEN = 40

contador = Counter()
for texto in df_train[COLUMNA_TEXTO]:
    contador.update(tokenizar(texto))

vocab = ['<pad>', '<unk>'] + [w for w, _ in contador.most_common(MAX_VOCAB - 2)]
word2id = {w: i for i, w in enumerate(vocab)}
PAD_ID, UNK_ID = 0, 1

def codificar(texto):
    ids = [word2id.get(w, UNK_ID) for w in tokenizar(texto)][:MAX_LEN]
    ids = ids + [PAD_ID] * (MAX_LEN - len(ids))
    return ids

class ComentariosDataset(Dataset):
    def __init__(self, df):
        self.X = torch.tensor([codificar(t) for t in df[COLUMNA_TEXTO]], dtype=torch.long)
        self.y = torch.tensor([label2id[l] for l in df['label']], dtype=torch.long)
    def __len__(self):
        return len(self.y)
    def __getitem__(self, idx):
        return self.X[idx], self.y[idx]

train_ds = ComentariosDataset(df_train)
val_ds = ComentariosDataset(df_val_interna)
test_ds = ComentariosDataset(df_test)

train_loader = DataLoader(train_ds, batch_size=256, shuffle=True)
val_loader = DataLoader(val_ds, batch_size=256)
test_loader = DataLoader(test_ds, batch_size=256)

conteo_clases = Counter(df_train['label'])
pesos = torch.tensor([len(df_train) / (len(EMOCIONES) * conteo_clases[e]) for e in EMOCIONES], dtype=torch.float32).to(DEVICE)

def entrenar_modelo(modelo, epochs=6, lr=1e-3):
    modelo.to(DEVICE)
    optimizador = torch.optim.Adam(modelo.parameters(), lr=lr)
    criterio = nn.CrossEntropyLoss(weight=pesos)
    mejor_f1, mejor_estado = 0, None

    for epoch in range(epochs):
        modelo.train()
        for xb, yb in train_loader:
            xb, yb = xb.to(DEVICE), yb.to(DEVICE)
            optimizador.zero_grad()
            loss = criterio(modelo(xb), yb)
            loss.backward()
            optimizador.step()

        modelo.eval()
        preds, reales = [], []
        with torch.no_grad():
            for xb, yb in val_loader:
                salida = modelo(xb.to(DEVICE))
                preds.extend(salida.argmax(1).cpu().tolist())
                reales.extend(yb.tolist())
        f1 = f1_score(reales, preds, average='macro', zero_division=0)
        print(f"Epoch {epoch+1}/{epochs} -- F1 macro (val interna 18k): {f1:.4f}")
        if f1 > mejor_f1:
            mejor_f1, mejor_estado = f1, {k: v.clone() for k, v in modelo.state_dict().items()}

    modelo.load_state_dict(mejor_estado)
    return modelo, mejor_f1

def predecir(modelo, loader):
    modelo.eval()
    preds = []
    with torch.no_grad():
        for xb, _ in loader:
            preds.extend(modelo(xb.to(DEVICE)).argmax(1).cpu().tolist())
    return [id2label[i] for i in preds]

def tamano_state_dict_mb(modelo):
    import io
    buffer = io.BytesIO()
    torch.save(modelo.state_dict(), buffer)
    return buffer.getbuffer().nbytes / (1024 ** 2)

Usando: cuda


## 8. CNN

In [8]:
class CNNClasificador(nn.Module):
    def __init__(self, vocab_size, embed_dim=128, num_classes=len(EMOCIONES), kernel_sizes=(3, 4, 5), num_filters=100):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, embed_dim, padding_idx=PAD_ID)
        self.convs = nn.ModuleList([nn.Conv1d(embed_dim, num_filters, k) for k in kernel_sizes])
        self.dropout = nn.Dropout(0.3)
        self.fc = nn.Linear(num_filters * len(kernel_sizes), num_classes)

    def forward(self, x):
        x = self.embedding(x).transpose(1, 2)
        x = [torch.relu(conv(x)).max(dim=2).values for conv in self.convs]
        x = torch.cat(x, dim=1)
        return self.fc(self.dropout(x))

if torch.cuda.is_available():
    torch.cuda.reset_peak_memory_stats()

cnn = CNNClasificador(vocab_size=len(vocab))
cnn, f1_cnn_val = entrenar_modelo(cnn)

inicio_inf = time.time()
pred_cnn = predecir(cnn, test_loader)
latencia_cnn = (time.time() - inicio_inf) / len(df_test)
memoria_cnn = torch.cuda.max_memory_allocated() / (1024 ** 2) if torch.cuda.is_available() else None

print("=== CNN (split oficial, evaluado sobre las 10k reservadas) ===")
registrar_resultado("CNN", y_test, pred_cnn, latencia_prom_seg=latencia_cnn,
                     memoria_pico_mb=memoria_cnn, almacenamiento_modelo_mb=tamano_state_dict_mb(cnn))

Epoch 1/6 -- F1 macro (val interna 18k): 0.4608
Epoch 2/6 -- F1 macro (val interna 18k): 0.4508
Epoch 3/6 -- F1 macro (val interna 18k): 0.5113
Epoch 4/6 -- F1 macro (val interna 18k): 0.5439
Epoch 5/6 -- F1 macro (val interna 18k): 0.5537
Epoch 6/6 -- F1 macro (val interna 18k): 0.5553
=== CNN (split oficial, evaluado sobre las 10k reservadas) ===
              precision    recall  f1-score   support

     alegria       0.81      0.64      0.71      3282
    tristeza       0.48      0.76      0.59       479
       enojo       0.69      0.66      0.67      3104
       miedo       0.21      0.50      0.30        26
    sorpresa       0.43      0.60      0.50       409
       burla       0.79      0.72      0.75      1633
     neutral       0.39      0.55      0.45      1067

    accuracy                           0.65     10000
   macro avg       0.54      0.63      0.57     10000
weighted avg       0.69      0.65      0.66     10000

Guardado en 08_resultados_clasicos.csv. Modelos regi

## 9. RNN (BiLSTM)

In [9]:
class RNNClasificador(nn.Module):
    def __init__(self, vocab_size, embed_dim=128, hidden_dim=128, num_classes=len(EMOCIONES)):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, embed_dim, padding_idx=PAD_ID)
        self.lstm = nn.LSTM(embed_dim, hidden_dim, batch_first=True, bidirectional=True)
        self.dropout = nn.Dropout(0.3)
        self.fc = nn.Linear(hidden_dim * 2, num_classes)

    def forward(self, x):
        emb = self.embedding(x)
        salida, _ = self.lstm(emb)
        pooled = salida.mean(dim=1)
        return self.fc(self.dropout(pooled))

if torch.cuda.is_available():
    torch.cuda.reset_peak_memory_stats()

rnn = RNNClasificador(vocab_size=len(vocab))
rnn, f1_rnn_val = entrenar_modelo(rnn)

inicio_inf = time.time()
pred_rnn = predecir(rnn, test_loader)
latencia_rnn = (time.time() - inicio_inf) / len(df_test)
memoria_rnn = torch.cuda.max_memory_allocated() / (1024 ** 2) if torch.cuda.is_available() else None

print("=== RNN / BiLSTM (split oficial, evaluado sobre las 10k reservadas) ===")
registrar_resultado("RNN (BiLSTM)", y_test, pred_rnn, latencia_prom_seg=latencia_rnn,
                     memoria_pico_mb=memoria_rnn, almacenamiento_modelo_mb=tamano_state_dict_mb(rnn))

Epoch 1/6 -- F1 macro (val interna 18k): 0.4350
Epoch 2/6 -- F1 macro (val interna 18k): 0.4954
Epoch 3/6 -- F1 macro (val interna 18k): 0.4741
Epoch 4/6 -- F1 macro (val interna 18k): 0.5316
Epoch 5/6 -- F1 macro (val interna 18k): 0.5385
Epoch 6/6 -- F1 macro (val interna 18k): 0.5383
=== RNN / BiLSTM (split oficial, evaluado sobre las 10k reservadas) ===
              precision    recall  f1-score   support

     alegria       0.77      0.66      0.71      3282
    tristeza       0.52      0.71      0.60       479
       enojo       0.67      0.63      0.65      3104
       miedo       0.16      0.46      0.24        26
    sorpresa       0.34      0.62      0.43       409
       burla       0.85      0.67      0.75      1633
     neutral       0.33      0.48      0.39      1067

    accuracy                           0.63     10000
   macro avg       0.52      0.60      0.54     10000
weighted avg       0.68      0.63      0.65     10000

Guardado en 08_resultados_clasicos.csv. Mod

## 10. GNN (TextGCN simplificado)

In [10]:
import scipy.sparse as sp
from collections import defaultdict

df_grafo = pd.concat([df_train, df_val_interna, df_test], ignore_index=True)
textos_grafo = df_grafo[COLUMNA_TEXTO].tolist()
etiquetas_grafo = df_grafo['label'].tolist()
n_docs = len(df_grafo)

particion_grafo = np.array(["train"] * len(df_train) + ["val"] * len(df_val_interna) + ["test"] * len(df_test))

print(f"Documentos en el grafo: {n_docs} (train={len(df_train)}, val_interna={len(df_val_interna)}, test={len(df_test)})")

tokens_grafo = [tokenizar(t) for t in textos_grafo]

MAX_VOCAB_GNN = 6000
contador_gnn = Counter()
for toks in tokens_grafo:
    contador_gnn.update(toks)
vocab_gnn = [w for w, _ in contador_gnn.most_common(MAX_VOCAB_GNN)]
word2idx_gnn = {w: i for i, w in enumerate(vocab_gnn)}
n_words = len(vocab_gnn)
n_nodes = n_docs + n_words
print(f"Nodos del grafo: {n_docs} documentos + {n_words} palabras = {n_nodes}")

tfidf_gnn = TfidfVectorizer(vocabulary=word2idx_gnn, tokenizer=tokenizar, lowercase=False, token_pattern=None)
X_tfidf_gnn = tfidf_gnn.fit_transform(textos_grafo)

co_ocurrencia = defaultdict(int)
frecuencia_palabra = defaultdict(int)
total_ventanas = 0

for toks in tokens_grafo:
    ids_presentes = sorted(set(word2idx_gnn[w] for w in toks if w in word2idx_gnn))
    if not ids_presentes:
        continue
    total_ventanas += 1
    for w in ids_presentes:
        frecuencia_palabra[w] += 1
    for i in range(len(ids_presentes)):
        for j in range(i + 1, len(ids_presentes)):
            co_ocurrencia[(ids_presentes[i], ids_presentes[j])] += 1

filas_ww, cols_ww, datos_ww = [], [], []
for (i, j), veces in co_ocurrencia.items():
    p_ij = veces / total_ventanas
    p_i = frecuencia_palabra[i] / total_ventanas
    p_j = frecuencia_palabra[j] / total_ventanas
    pmi = np.log(p_ij / (p_i * p_j))
    if pmi > 0:
        filas_ww += [i, j]
        cols_ww += [j, i]
        datos_ww += [pmi, pmi]

A_ww = sp.coo_matrix((datos_ww, (filas_ww, cols_ww)), shape=(n_words, n_words))
print(f"Aristas palabra-palabra (PMI > 0): {len(datos_ww) // 2}")

A_dw = sp.csr_matrix(X_tfidf_gnn)
superior = sp.hstack([sp.csr_matrix((n_docs, n_docs)), A_dw])
inferior = sp.hstack([A_dw.T, A_ww.tocsr()])
A = sp.vstack([superior, inferior]).tocsr()
A = A + sp.eye(n_nodes)

grados = np.array(A.sum(axis=1)).flatten()
D_inv_sqrt = sp.diags(1.0 / np.sqrt(grados))
A_norm = D_inv_sqrt @ A @ D_inv_sqrt

def sparse_a_tensor(mat):
    mat = mat.tocoo()
    indices = torch.tensor(np.vstack([mat.row, mat.col]), dtype=torch.long)
    valores = torch.tensor(mat.data, dtype=torch.float32)
    return torch.sparse_coo_tensor(indices, valores, mat.shape).coalesce()

A_norm_t = sparse_a_tensor(A_norm).to(DEVICE)

Documentos en el grafo: 100000 (train=72000, val_interna=18000, test=10000)
Nodos del grafo: 100000 documentos + 6000 palabras = 106000
Aristas palabra-palabra (PMI > 0): 1814795


/tmp/ipykernel_1344/2569022961.py:74: UserWarning: Sparse invariant checks are implicitly disabled. Memory errors (e.g. SEGFAULT) will occur when operating on a sparse tensor which violates the invariants, but checks incur performance overhead. To silence this warning, explicitly opt in or out. See `torch.sparse.check_sparse_tensor_invariants.__doc__` for guidance.  (Triggered internally at /__w/pytorch/pytorch/aten/src/ATen/Context.cpp:848.)
  return torch.sparse_coo_tensor(indices, valores, mat.shape).coalesce()


In [11]:
class GCNCapa(nn.Module):
    def __init__(self, in_dim, out_dim):
        super().__init__()
        self.peso = nn.Parameter(torch.empty(in_dim, out_dim))
        nn.init.xavier_uniform_(self.peso)

    def forward(self, A_norm, X, X_es_sparsa=False):
        soporte = torch.sparse.mm(X, self.peso) if X_es_sparsa else X @ self.peso
        return torch.sparse.mm(A_norm, soporte)

class GCN(nn.Module):
    def __init__(self, n_nodes, hidden_dim=100, num_classes=len(EMOCIONES)):
        super().__init__()
        self.capa1 = GCNCapa(n_nodes, hidden_dim)
        self.capa2 = GCNCapa(hidden_dim, num_classes)
        self.dropout = nn.Dropout(0.5)

    def forward(self, A_norm, X):
        h = torch.relu(self.capa1(A_norm, X, X_es_sparsa=True))
        h = self.dropout(h)
        return self.capa2(A_norm, h, X_es_sparsa=False)

X_identidad = sparse_a_tensor(sp.eye(n_nodes)).to(DEVICE)

if torch.cuda.is_available():
    torch.cuda.reset_peak_memory_stats()

gcn = GCN(n_nodes=n_nodes).to(DEVICE)
optimizador_gcn = torch.optim.Adam(gcn.parameters(), lr=0.01, weight_decay=5e-4)

y_grafo = torch.tensor([label2id[e] for e in etiquetas_grafo], dtype=torch.long).to(DEVICE)
idx_train_gnn = torch.tensor(np.where(particion_grafo == "train")[0], dtype=torch.long).to(DEVICE)
idx_val_gnn = torch.tensor(np.where(particion_grafo == "val")[0], dtype=torch.long).to(DEVICE)
idx_test_gnn = torch.tensor(np.where(particion_grafo == "test")[0], dtype=torch.long).to(DEVICE)

conteo_gnn = Counter(df_train['label'])
pesos_balanceados = [len(df_train) / (len(EMOCIONES) * max(1, conteo_gnn[e])) for e in EMOCIONES]
TOPE_PESO = 10.0
pesos_gnn = torch.tensor([min(p, TOPE_PESO) for p in pesos_balanceados], dtype=torch.float32).to(DEVICE)
criterio_gnn = nn.CrossEntropyLoss(weight=pesos_gnn)

mejor_f1_gnn, mejor_estado_gnn = 0, None
EPOCHS_GNN = 100

for epoch in range(EPOCHS_GNN):
    gcn.train()
    optimizador_gcn.zero_grad()
    salida = gcn(A_norm_t, X_identidad)
    loss = criterio_gnn(salida[idx_train_gnn], y_grafo[idx_train_gnn])
    loss.backward()
    optimizador_gcn.step()

    if (epoch + 1) % 10 == 0:
        gcn.eval()
        with torch.no_grad():
            salida = gcn(A_norm_t, X_identidad)
            preds_val = salida[idx_val_gnn].argmax(1).cpu().numpy()
        f1 = f1_score(y_grafo[idx_val_gnn].cpu().numpy(), preds_val, average='macro', zero_division=0)
        print(f"Epoch {epoch+1}/{EPOCHS_GNN} -- loss: {loss.item():.4f} -- F1 macro (val interna 18k): {f1:.4f}")
        if f1 > mejor_f1_gnn:
            mejor_f1_gnn, mejor_estado_gnn = f1, {k: v.clone() for k, v in gcn.state_dict().items()}

gcn.load_state_dict(mejor_estado_gnn)
gcn.eval()

inicio_inf = time.time()
with torch.no_grad():
    salida = gcn(A_norm_t, X_identidad)
    preds_test_gnn = salida[idx_test_gnn].argmax(1).cpu().numpy()
latencia_gnn = (time.time() - inicio_inf) / len(idx_test_gnn)
memoria_gnn = torch.cuda.max_memory_allocated() / (1024 ** 2) if torch.cuda.is_available() else None

pred_gnn = [id2label[i] for i in preds_test_gnn]
y_test_gnn_real = [id2label[i] for i in y_grafo[idx_test_gnn].cpu().numpy()]

print("=== GNN (TextGCN-lite, split oficial completo -- 10k de prueba reservadas) ===")
registrar_resultado(
    "GNN", y_test_gnn_real, pred_gnn,
    latencia_prom_seg=latencia_gnn, memoria_pico_mb=memoria_gnn,
    almacenamiento_modelo_mb=tamano_state_dict_mb(gcn),
)

Epoch 10/100 -- loss: 1.9458 -- F1 macro (val interna 18k): 0.0795
Epoch 20/100 -- loss: 1.9459 -- F1 macro (val interna 18k): 0.0509
Epoch 30/100 -- loss: 1.9459 -- F1 macro (val interna 18k): 0.1076
Epoch 40/100 -- loss: 1.9459 -- F1 macro (val interna 18k): 0.1130
Epoch 50/100 -- loss: 1.9459 -- F1 macro (val interna 18k): 0.0548
Epoch 60/100 -- loss: 1.9459 -- F1 macro (val interna 18k): 0.0300
Epoch 70/100 -- loss: 1.9459 -- F1 macro (val interna 18k): 0.0662
Epoch 80/100 -- loss: 1.9459 -- F1 macro (val interna 18k): 0.0645
Epoch 90/100 -- loss: 1.9459 -- F1 macro (val interna 18k): 0.0948
Epoch 100/100 -- loss: 1.9459 -- F1 macro (val interna 18k): 0.0337
=== GNN (TextGCN-lite, split oficial completo -- 10k de prueba reservadas) ===
              precision    recall  f1-score   support

     alegria       0.36      0.12      0.18      3282
    tristeza       0.08      0.00      0.01       479
       enojo       0.32      0.08      0.13      3104
       miedo       0.00      0.00

## Guardar `08_resultados_clasicos.csv` en Hugging Face

In [12]:
from huggingface_hub import upload_file

upload_file(path_or_fileobj=RUTA_RESULTADOS, path_in_repo=f"final/{RUTA_RESULTADOS}",
            repo_id=REPO_DATOS, repo_type="dataset")
print(f"Subido a Hugging Face: final/{RUTA_RESULTADOS}")

Subido a Hugging Face: final/08_resultados_clasicos.csv
